# 04 — Re-identification

Purpose: controlled Stage 1 re-identification experiment. Required inputs: reviewed frame/crop cache and exact local assets; see config and MODEL_INVENTORY.

OSNet is directly adapted through BoxMOT. PRTReID global and full part+visibility, and BPBReID part+visibility remain separate conditions in isolated legacy upstream environments. See upstream extraction specification; unavailable bridges are blocked. Use one immutable crop manifest for every method. GT crop retrieval measures representation quality; predicted-track crops separately measure deployment recovery. Never mix their scores. Shared crop files preserve pixel content; each upstream method retains its documented normalization. Query/gallery roles and tracklets must be disjoint. Compare same-team, view, occlusion, size and time-gap subsets; sample sizes must accompany results.

No results have been executed. GPU checks and opt-in flags precede model loading.

In [ ]:
from pathlib import Path
import sys, json, os, subprocess
# Attach the unpacked laboratory as a Kaggle Dataset. Set LAB_ROOT if discovery is ambiguous.
LAB_ROOT = None
candidates = sorted({p.parent for p in Path('/kaggle/input').rglob('README.md') if (p.parent/'scoutra_lab'/'common.py').is_file()}) if Path('/kaggle/input').exists() else []
if LAB_ROOT is None:
    if len(candidates) != 1: raise RuntimeError(f'Set LAB_ROOT to the attached laboratory folder. Candidates: {candidates}')
    LAB_ROOT = candidates[0]
LAB_ROOT = Path(LAB_ROOT)
assert (LAB_ROOT/'scoutra_lab'/'common.py').is_file()
sys.path.insert(0, str(LAB_ROOT))
WORK = Path('/kaggle/working/scoutra')
WORK.mkdir(parents=True, exist_ok=True)
print('Laboratory:', LAB_ROOT, 'Outputs:', WORK)


In [ ]:
INSTALL = False
PROFILE = 'tracking-reid.txt'
if INSTALL:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(LAB_ROOT/'configs'/'profiles'/PROFILE)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'check'])
    (WORK/'requirements-resolved.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
else:
    print('Installation skipped. Direct pins are provisional until pip check passes in this Kaggle image.')


## Check assets and device
Attach weights as a separate private Dataset only after reviewing the exact source/model terms. Populate `checkpoint.path`, expected SHA256, `license_reviewed`, and the raw class map where applicable. Directory checkpoints use the fingerprint defined in `common.asset_fingerprint`. The notebook does not download weights. Do not invent a Scoutra baseline when its checkpoint is unavailable.

One selected T4 runs one model. A second T4 may run an independent job after isolation/timing is verified. Never report concurrently contended runtime as an isolated benchmark. Initial runs use batch one; no automatic distributed model sharding.

In [ ]:
from scoutra_lab.common import load_config, environment
from scoutra_lab.pipelines import reid
CFG = load_config(LAB_ROOT/'configs/reid_osnet.yaml')
# Edit a copied config explicitly. Set a NEW experiment_id for every run.
# CFG['model']['checkpoint'] = {'path': 'VERIFIED_LOCAL_PATH', 'sha256': 'VERIFIED_SHA256', 'license_reviewed': True}
RUN_INFERENCE = False
CFG['execute'] = RUN_INFERENCE
print('Execution enabled:', CFG['execute'])
print('Method configuration:', CFG)
# environment() records package versions/device inventory without inference.
print(environment().get('devices', []))


## Prepare one crop manifest before extraction
Use `scoutra_lab.crops.make_crops` on reviewed GT boxes for representation testing, or the frozen detection/track cache for deployment testing. See notebook code below. Assign reviewed `source_match`, `player_id`, `tracklet_id`, `retrieval_role` and condition fields in the manifest before extraction. Ground-truth labels are only used for evaluation, not to choose embeddings or thresholds on test matches.

In [ ]:
RUN_PREPARE_CROPS = False
if RUN_PREPARE_CROPS:
    from scoutra_lab.dataset import load_frames
    from scoutra_lab.common import read_table
    from scoutra_lab.crops import make_crops
    f = load_frames(WORK/'cache/stage1-v1/frames.json')
    # Predicted track crop condition; for representation GT crops, load reviewed GT objects instead.
    boxes = read_table(WORK/'results/B-bytetrack-001/predictions.parquet')
    crops = make_crops(f, boxes, WORK/'crops/reid-v1')
    print('Review/label before evaluating:', len(crops))


In [ ]:
# With RUN_INFERENCE=False this creates a NOT EXECUTED record only.
# Requires base serialization dependencies; no model is loaded.
RESULT = reid(CFG)
print('Result directory:', RESULT)
print((RESULT/'metrics.json').read_text())
print((RESULT/'runtime.json').read_text())


## Evaluation, runtime and error review
Open `metrics.json` only after checking execution/evaluation status. Keep quality on easy and hard conditions separate; compare matched thresholds and independent matches. Runtime synchronizes CUDA and records allocated/reserved peaks on the selected GPU; it excludes initial loading and includes disk/decode, preprocessing and serialization as specified in runtime.json. Add separate cold-load/warm steady-state measurements in later runs; no FPS value is prefilled.

Use `scoutra_lab.visualize.render` with the shared frames and result rows to save labeled overlays. Review identity-loss/switch timestamps against GT. Save selected-frame failures and manually reviewed error categories. Quantitative conclusions need confidence intervals and a practical margin; fewer than five independent matches remain descriptive. See EXPERIMENT_PLAN for preregistered progression gates.

## Exact weights and full-part distinction
OSNet asset selected in BoxMOT16.0.10's registry: `osnet_x0_25_msmt17.pt`, [listed Drive source](https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF); weight redistribution/data grant not independently cleared. [PRT source30617a7](https://github.com/VlSomers/prtreid/tree/30617a75967e84d5d516959c4b84cbeea6f56493), [SoccerNet checkpoint archive](https://zenodo.org/records/10653453). [BPB sourcea2dc430](https://github.com/VlSomers/bpbreid/tree/a2dc4304284784d8b2c061764512f0698aa38c21), [BPB/HRNet archive](https://zenodo.org/records/10604211). Sources use Hippocratic terms; archive metadata differs (CC-BY4.0).

The inspected GSR default uses `test_embeddings: ['globl']`. Full parts must retain part vectors and visibility and use the upstream distance routine; they are separate experimental conditions. See MODEL_INVENTORY for exact tuple/API differences and the **blocked legacy bridge specification**. The notebook's direct runner executes OSNet only and refuses PRT/BPB surrogates. Query/gallery Rank1/mAP are automated; a fixed validation-chosen `validation_distance_threshold` additionally reports TPR/FMR. Stratified part-aware recovery still requires the validated upstream bridge.

## Outputs and troubleshooting
Each new experiment ID creates `config.yaml`, `metadata.json`, `metrics.json`, `predictions.parquet`, `runtime.json`, `errors.json`, `report.md` and `visualizations/` below `/kaggle/working/scoutra/results/`. Existing IDs are never overwritten. Unexecuted predictions are an empty **schema container**, not measured empty detections; inspect status first. Missing annotations leave metrics null.

Download the entire results directory using Kaggle **Save Version → Save & Run All** only after flags are intentional, then download notebook outputs. Also save manifest, cache configuration, annotations, hashes and resolved environment. Do not enable all notebooks or all videos together.

- Missing `/kaggle/input/...`: print the mount contents and set the verified root. Dataset titles and mount paths can differ.
- No CUDA / unsupported GPU: stop; never silently move inference to this cloud or a local machine.
- Out of memory: reduce batch/frame/pixel budget in a new recorded configuration. The second T4 is not pooled memory.
- Missing/hash-mismatched weight: attach the exact reviewed checkpoint; do not auto-fetch a similarly named model.
- Dependency/ABI error: use a fresh Kaggle session and the method-specific profile. Preserve the failed `pip check`/freeze log. Do not upgrade all packages in place.
- Missing metrics: label and review the required frames. Visual inspection is not an accuracy result.
- Source/package API mismatch: preserve the error and version; do not patch the algorithm silently.

No execution or benchmark results are embedded in this notebook.
